# YouTube → SRT (ruso) con faster-whisper

Notebook autocontenido. No depende del resto del repo Sta-RU.

**Qué hace:** descarga audio de una lista de links de YouTube, los transcribe a SRT en ruso con `large-v2`, y te entrega todo en un ZIP. Suena un ruidito cuando termina.

**Antes de correr:** `Runtime → Change runtime type → T4 GPU` (o cualquier GPU). Sin GPU también corre, pero mucho más lento.


## 1) Setup

In [ ]:
!pip install -q faster-whisper yt-dlp deep-translator
!apt-get -qq install -y ffmpeg > /dev/null

import torch, os, shutil, zipfile, time, re
import yt_dlp
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "int8_float16" if DEVICE == "cuda" else "int8"
print(f"Device: {DEVICE}  |  compute_type: {COMPUTE_TYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

YTDLP_CLIENTS = ["default", "tv_simply", "mweb", "android_vr", "tv", "web_safari"]

def _youtube_reachable():
    canary = "https://www.youtube.com/watch?v=jNQXAC9IVRw"
    for client in YTDLP_CLIENTS:
        opts = {"quiet": True, "no_warnings": True, "skip_download": True, "simulate": True}
        if client != "default":
            opts["extractor_args"] = {"youtube": {"player_client": [client]}}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                if ydl.extract_info(canary, download=False):
                    return True
        except Exception:
            continue
    return False

print()
if _youtube_reachable():
    print("\033[1;92mALL OK\033[0m")
else:
    print("\033[1;91mYOUTUBE IS BLOCKING YOU, CHANGE YOUR IP\033[0m")


## 2) Pegá tus links de YouTube

Corré esta celda y aparece un **cuadro de texto**. Pegá ahí tus links (uno por línea) y seguí con la celda 3 — no hace falta tocar código.

Admite videos sueltos y playlists. Las líneas que arrancan con `#` se ignoran.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

url_box = widgets.Textarea(
    value="",
    placeholder="Pegá acá tus links de YouTube, uno por línea:\nhttps://www.youtube.com/watch?v=...\nhttps://www.youtube.com/watch?v=...",
    layout=widgets.Layout(width="95%", height="180px"),
    continuous_update=True,
)
display(url_box)
print("↑ Pegá tus links en el cuadro (uno por línea) y después corré la celda 3.")


## 3) Descargar audio

Descarga el audio y arma el nombre de salida: traduce el título al inglés y nombra cada SRT como `Fecha - Título en inglés-RU.srt`.

Además:
- **Deduplica** links repetidos (mismo video pegado dos veces o repetido en una playlist) → se procesa una sola vez.
- Si dos videos **distintos** terminan con el mismo nombre (misma fecha + título traducido idéntico), les agrega el id de YouTube entre corchetes solo a los que chocan, para que ninguno pise al otro.


In [ ]:
from collections import Counter
from deep_translator import GoogleTranslator

# Leer los links del cuadro de la celda 2
URLS = [u.strip() for u in url_box.value.strip().splitlines() if u.strip() and not u.strip().startswith("#")]
assert URLS, "El cuadro de links está vacío. Volvé a la celda 2, pegá los links y corré esta celda otra vez."
print(f"{len(URLS)} link(s) a procesar:")
for u in URLS:
    print("  -", u)
print()

AUDIO_DIR = Path("/content/audios")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MEDIA_EXTS = {".m4a", ".webm", ".opus", ".mp3", ".mp4", ".wav", ".ogg", ".aac", ".mkv"}

_FORBIDDEN = re.compile(r'[\\/:*?"<>|\n\r\t]')
def sanitize(name):
    return _FORBIDDEN.sub("_", name).strip().rstrip(". ") or "untitled"

def fmt_date(raw):
    if raw and len(raw) == 8 and raw.isdigit():
        return f"{raw[:4]}-{raw[4:6]}-{raw[6:]}"
    return raw or ""

def ydl_run(url, download):
    """extract_info ciclando player clients para esquivar el bot-check de IP cloud."""
    last = "unknown"
    for client in YTDLP_CLIENTS:
        opts = {
            "outtmpl": str(AUDIO_DIR / "%(id)s.%(ext)s"),
            "format": "bestaudio[ext=m4a]/bestaudio/best",
            "ignoreerrors": True, "quiet": True, "no_warnings": True,
        }
        if client != "default":
            opts["extractor_args"] = {"youtube": {"player_client": [client]}}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(url, download=download)
            if info:
                return info
            last = "no info (posible bot-check)"
        except Exception as ex:
            last = str(ex)
    raise RuntimeError(f"yt-dlp falló para {url}: {last}")

translator = GoogleTranslator(source="auto", target="en")
def translate_en(title):
    if not title:
        return ""
    try:
        return (translator.translate(title) or title).strip()
    except Exception as ex:
        print(f"   [WARN] no pude traducir el título ({ex}); uso el original")
        return title

# 1) Descargar y recolectar entradas, deduplicando por video id
#    (si pegaste el mismo link dos veces, o una playlist lo repite, se procesa una sola vez)
entries = {}  # vid -> {audio, title, date}
for url in URLS:
    info = ydl_run(url, download=True)
    for ent in (info.get("entries") or [info]):
        if not ent:
            continue
        vid = ent.get("id")
        if not vid or vid in entries:
            continue
        matches = sorted(p for p in AUDIO_DIR.glob(f"{vid}.*") if p.suffix.lower() in MEDIA_EXTS)
        if not matches:
            print(f"   [WARN] no encontré el audio descargado para id={vid}")
            continue
        entries[vid] = {
            "audio": matches[0],
            "title": ent.get("title", "") or "",
            "date": fmt_date(ent.get("upload_date", "")),
        }

# 2) Traducir títulos y armar el stem base "Fecha - Título en inglés"
for vid, e in entries.items():
    title_en = translate_en(e["title"])
    e["stem"] = " - ".join(p for p in [e["date"], title_en] if p) or vid

# 3) Garantizar nombres únicos: si dos videos DISTINTOS comparten el mismo stem
#    (misma fecha + título traducido idéntico), les agrego el id de YouTube solo
#    a los que chocan. Sin esto, el segundo pisaría al primero y se perdería al
#    saltar "ya existe" en la celda 4.
stem_counts = Counter(e["stem"] for e in entries.values())

jobs = []  # cada job: {'audio': Path, 'srt': Path}
for vid, e in entries.items():
    stem = e["stem"] if stem_counts[e["stem"]] == 1 else f"{e['stem']} [{vid}]"
    jobs.append({"audio": e["audio"], "srt": AUDIO_DIR / f"{sanitize(stem)}-RU.srt"})

print(f"\nDescargados: {len(jobs)}")
for j in jobs:
    print("  -", j["srt"].name)


## 4) Transcribir a SRT (ruso, large-v2)

Parámetros equivalentes a tu llamada de PowerShell:

| PowerShell (faster-whisper-xxl) | Python (faster-whisper) |
|---|---|
| `--model large-v2` | `WhisperModel("large-v2")` |
| `--language ru` | `language="ru"` |
| `--compute_type int8_float16` | `compute_type="int8_float16"` |
| `--temperature 0` | `temperature=0` |
| `--beam_size 5` | `beam_size=5` |
| `--best_of 1` | `best_of=1` |
| `--task transcribe` | `task="transcribe"` |
| `--max_line_count 1 --sentence` | una línea por segmento al escribir SRT |
| `--max_line_width 200` | no se trunca (segmentos enteros) |

Si el SRT ya existe, salta (igual que tu script de PowerShell).


In [ ]:
from faster_whisper import WhisperModel

def fmt_ts(t):
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    if ms == 1000:
        ms = 0
        s += 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

print("Cargando modelo large-v2...")
model = WhisperModel("large-v2", device=DEVICE, compute_type=COMPUTE_TYPE)
print("Modelo listo.\n")

total = len(jobs)
t_global = time.time()
for i, job in enumerate(jobs, 1):
    audio, srt = job["audio"], job["srt"]
    if srt.exists():
        print(f"[{i}/{total}] SALTADO (ya existe): {srt.name}")
        continue

    print(f"[{i}/{total}] Procesando: {srt.name}")
    t0 = time.time()
    segments, info = model.transcribe(
        str(audio),
        language="ru",
        task="transcribe",
        temperature=0,
        beam_size=5,
        best_of=1,
        vad_filter=True,
        condition_on_previous_text=True,
    )
    print(f"   duración audio: {info.duration:.1f}s")

    n = 0
    with open(srt, "w", encoding="utf-8") as f:
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            n += 1
            f.write(f"{n}\n{fmt_ts(seg.start)} --> {fmt_ts(seg.end)}\n{text}\n\n")
    dt = time.time() - t0
    print(f"[{i}/{total}] Listo: {srt.name}  ({n} líneas, {dt:.1f}s)\n")

print(f"Finalizado. {total} videos procesados en {(time.time()-t_global)/60:.1f} min.")


## 5) Empaquetar SRTs en ZIP + ruidito + descarga

In [ ]:
from IPython.display import Audio, display
import numpy as np
from google.colab import files

# Empaquetar exactamente los SRT de esta corrida (los de `jobs`), no un glob del
# directorio: así no se cuela ningún .srt viejo de una corrida anterior.
srts = [j["srt"] for j in jobs if j["srt"].exists()]
missing = [j["srt"].name for j in jobs if not j["srt"].exists()]
print(f"SRTs a empaquetar: {len(srts)} de {len(jobs)}")
for s in srts:
    print("  -", s.name)
if missing:
    print("\n[WARN] faltan estos SRT (¿falló o no corrió la transcripción?):")
    for m in missing:
        print("  -", m)

ZIP_PATH = "/content/subtitulos_ru.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for s in srts:
        zf.write(s, arcname=s.name)
print(f"\nZIP listo: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1024:.1f} KB)")

# Ruidito: fanfarria ascendente sol-do-mi-sol-do
sr = 22050
out = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out = np.concatenate([out, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out, rate=sr, autoplay=True))

# Disparar la descarga al navegador
files.download(ZIP_PATH)
